In [1]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              accuracy_score, classification_report,
                              confusion_matrix, precision_recall_curve,
                              roc_curve, auc)
from sklearn.feature_selection import SelectKBest, chi2
from scipy.sparse import hstack
import gc

In [2]:
# Create output directories
os.makedirs("mbert-28dec/confusion_matrices", exist_ok=True)
os.makedirs("mbert-28dec/pr_curves", exist_ok=True)
os.makedirs("mbert-28dec/roc_curves", exist_ok=True)
os.makedirs("mbert-28dec/comparison", exist_ok=True)

In [3]:
# GPU Memory Management
def clear_gpu_memory():
    """Clear GPU cache and run garbage collection"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

In [4]:
def get_device():
    """Select best available device with memory check"""
    if torch.cuda.is_available():
        free_mem, total_mem = torch.cuda.mem_get_info()
        free_gb = free_mem / (1024 ** 3)
        print(f"GPU Available: {torch.cuda.get_device_name(0)}")
        print(f"Free VRAM: {free_gb:.2f} GB")
        return torch.device('cuda')
    print("Using CPU")
    return torch.device('cpu')

device = get_device()

GPU Available: Tesla T4
Free VRAM: 14.64 GB


In [5]:
# Load data
print("Loading datasets...")
with open('/content/train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open('/content/test.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

Loading datasets...


In [6]:
# Labels
train_df['Binary_Label'] = train_df['Label_Binary'].apply(lambda x: 1 if x == 'OFF' else 0)
test_df['Binary_Label'] = test_df['Label_Binary'].apply(lambda x: 1 if x == 'OFF' else 0)

label_map = {'NO': 0, 'OO': 1, 'OR': 2, 'OS': 3}
train_df['Multiclass_Label'] = train_df['Label_Multiclass'].apply(lambda x: label_map.get(x, 0))
test_df['Multiclass_Label'] = test_df['Label_Multiclass'].apply(lambda x: label_map.get(x, 0))

In [7]:
# Preprocessing
train_comments = train_df['Comment'].tolist()
test_comments = test_df['Comment'].tolist()

In [8]:
# TF-IDF Features (optimized with max_features)
print("Extracting TF-IDF features...")
char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1,5), max_features=5000)
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=5000)

X_train_char = char_vectorizer.fit_transform(train_comments)
X_test_char = char_vectorizer.transform(test_comments)
X_train_word = word_vectorizer.fit_transform(train_comments)
X_test_word = word_vectorizer.transform(test_comments)

X_train = hstack([X_train_word, X_train_char])
X_test = hstack([X_test_word, X_test_char])

Extracting TF-IDF features...


In [9]:
# Feature selection
selector = SelectKBest(chi2, k=min(5000, X_train.shape[1]))
X_train_selected = selector.fit_transform(X_train, train_df['Binary_Label'])
X_test_selected = selector.transform(X_test)

y_train_bin = train_df['Binary_Label']
y_test_bin = test_df['Binary_Label']
y_train_multi = train_df['Multiclass_Label']
y_test_multi = test_df['Multiclass_Label']

In [10]:
# Evaluation Functions
def plot_confusion_matrix(y_true, y_pred, labels, title, filename):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'mbert-28dec/confusion_matrices/{filename}.png', dpi=300, bbox_inches='tight')
    plt.close()

In [11]:
def plot_pr_curve(y_true, y_scores, title, filename):
    """Plot and save precision-recall curve"""
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, marker='.', linewidth=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'mbert-28dec/pr_curves/{filename}.png', dpi=300, bbox_inches='tight')
    plt.close()

In [12]:
def plot_roc_curve(y_true, y_scores, title, filename):
    """Plot and save ROC curve"""
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'mbert-28dec/roc_curves/{filename}.png', dpi=300, bbox_inches='tight')
    plt.close()

In [13]:
def evaluate_model(y_true, y_pred, model_name, labels, is_binary=True, y_scores=None):
    """Comprehensive model evaluation"""
    print(f"\n{'='*60}")
    print(f"{model_name} - {'Binary' if is_binary else 'Multiclass'} Classification")
    print('='*60)

    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

    # Confusion Matrix
    safe_name = model_name.replace(' ', '_').replace('/', '_')
    plot_confusion_matrix(y_true, y_pred, labels,
                         f'{model_name} Confusion Matrix',
                         f'{safe_name}_confusion')

    # PR and ROC curves (only for binary with scores)
    if is_binary and y_scores is not None:
        plot_pr_curve(y_true, y_scores,
                     f'{model_name} Precision-Recall Curve',
                     f'{safe_name}_pr')
        plot_roc_curve(y_true, y_scores,
                      f'{model_name} ROC Curve',
                      f'{safe_name}_roc')

    return acc

In [14]:
# Classical ML Models
print("\n" + "="*60)
print("TRAINING CLASSICAL ML MODELS")
print("="*60)


TRAINING CLASSICAL ML MODELS


In [15]:
results_binary = {}

In [16]:
# Logistic Regression
print("\nTraining Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_selected, y_train_bin)
pred_lr = lr.predict(X_test_selected)
scores_lr = lr.predict_proba(X_test_selected)[:, 1]
results_binary['LogisticRegression'] = evaluate_model(
    y_test_bin, pred_lr, 'Logistic Regression',
    ['Non-Offensive', 'Offensive'], y_scores=scores_lr
)


Training Logistic Regression...

Logistic Regression - Binary Classification
Accuracy: 0.7841

Classification Report:
               precision    recall  f1-score   support

Non-Offensive       0.78      0.91      0.84       896
    Offensive       0.80      0.58      0.67       554

     accuracy                           0.78      1450
    macro avg       0.79      0.74      0.76      1450
 weighted avg       0.79      0.78      0.78      1450



In [17]:
# SVM
print("\nTraining SVM...")
svm = SVC(kernel='linear', probability=True, random_state=42)
svm.fit(X_train_selected, y_train_bin)
pred_svm = svm.predict(X_test_selected)
scores_svm = svm.predict_proba(X_test_selected)[:, 1]
results_binary['SVM'] = evaluate_model(
    y_test_bin, pred_svm, 'SVM',
    ['Non-Offensive', 'Offensive'], y_scores=scores_svm
)


Training SVM...

SVM - Binary Classification
Accuracy: 0.7821

Classification Report:
               precision    recall  f1-score   support

Non-Offensive       0.79      0.89      0.83       896
    Offensive       0.77      0.61      0.68       554

     accuracy                           0.78      1450
    macro avg       0.78      0.75      0.76      1450
 weighted avg       0.78      0.78      0.78      1450



In [18]:
# Random Forest
print("\nTraining Random Forest...")
rf_bin = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_bin.fit(X_train_selected, y_train_bin)
pred_rf = rf_bin.predict(X_test_selected)
scores_rf = rf_bin.predict_proba(X_test_selected)[:, 1]
results_binary['RandomForest'] = evaluate_model(
    y_test_bin, pred_rf, 'Random Forest',
    ['Non-Offensive', 'Offensive'], y_scores=scores_rf
)


Training Random Forest...

Random Forest - Binary Classification
Accuracy: 0.8048

Classification Report:
               precision    recall  f1-score   support

Non-Offensive       0.79      0.93      0.85       896
    Offensive       0.83      0.61      0.70       554

     accuracy                           0.80      1450
    macro avg       0.81      0.77      0.78      1450
 weighted avg       0.81      0.80      0.80      1450



In [19]:
# Multiclass Random Forest
print("\nTraining Multiclass Random Forest...")
rf_multi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_multi.fit(X_train_selected, y_train_multi)
pred_rf_multi = rf_multi.predict(X_test_selected)
evaluate_model(
    y_test_multi, pred_rf_multi, 'Random Forest Multiclass',
    ['NO', 'OO', 'OR', 'OS'], is_binary=False
)


Training Multiclass Random Forest...

Random Forest Multiclass - Multiclass Classification
Accuracy: 0.7710

Classification Report:
              precision    recall  f1-score   support

          NO       0.78      0.93      0.85       896
          OO       0.76      0.57      0.65       486
          OR       0.62      0.16      0.26        49
          OS       0.00      0.00      0.00        19

    accuracy                           0.77      1450
   macro avg       0.54      0.42      0.44      1450
weighted avg       0.76      0.77      0.75      1450



0.7710344827586207

In [20]:
# Two-Step Random Forest
print("\nTraining Two-Step Random Forest...")
offensive_train_mask = y_train_bin.values == 1
rf_two_step = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_two_step.fit(X_train_selected[offensive_train_mask],
                y_train_multi.values[offensive_train_mask])

pred_two_step = np.zeros(len(y_test_multi), dtype=int)
offensive_test_pred_mask = pred_rf == 1
pred_two_step[offensive_test_pred_mask] = rf_two_step.predict(
    X_test_selected[offensive_test_pred_mask]
)
evaluate_model(
    y_test_multi, pred_two_step, 'Two-Step Random Forest',
    ['NO', 'OO', 'OR', 'OS'], is_binary=False
)


Training Two-Step Random Forest...

Two-Step Random Forest - Multiclass Classification
Accuracy: 0.7814

Classification Report:
              precision    recall  f1-score   support

          NO       0.79      0.93      0.85       896
          OO       0.76      0.59      0.66       486
          OR       0.64      0.37      0.47        49
          OS       0.00      0.00      0.00        19

    accuracy                           0.78      1450
   macro avg       0.55      0.47      0.50      1450
weighted avg       0.77      0.78      0.77      1450



0.7813793103448275

In [21]:
# mBERT Dataset (Memory-Optimized)
class NepaliDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):  # Reduced from 128
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [26]:
# mBERT Training (Optimized for 4GB VRAM)
print("\n" + "="*60)
print("TRAINING mBERT MODELS (Memory-Optimized)")
print("="*60)

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

def train_mbert_optimized(train_comments, train_labels, test_comments, test_labels,
                          num_labels, model_name, epochs=2):
    """Memory-optimized mBERT training"""

    clear_gpu_memory()

    # Create datasets with smaller batch size
    train_dataset = NepaliDataset(train_comments, train_labels.tolist(), tokenizer, max_len=64)
    test_dataset = NepaliDataset(test_comments, test_labels.tolist(), tokenizer, max_len=64)

    # Smaller batch size for 4GB VRAM
    batch_size = 8 if device.type == 'cuda' else 16
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=0,
                             pin_memory=False)

    # Load model with optimizations
    # Changed to torch.float32 for numerical stability to avoid NaN loss
    dtype = torch.float32 # Previously: torch.float16 if device.type == 'cuda' else torch.float32
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-multilingual-cased',
        num_labels=num_labels,
        torch_dtype=dtype
    )

    # Enable gradient checkpointing to save memory
    if device.type == 'cuda':
        model.gradient_checkpointing_enable()

    model.to(device)

    # Optimizer with gradient accumulation
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    accumulation_steps = 4  # Effective batch size = 4 * 4 = 16

    # Training loop
    print(f"\nTraining {model_name}...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        optimizer.zero_grad()

        for i, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

            total_loss += loss.item() * accumulation_steps

            # Clear cache periodically
            if i % 50 == 0 and device.type == 'cuda':
                torch.cuda.empty_cache()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

    # Evaluation
    model.eval()
    predictions = []
    probabilities = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            if num_labels == 2:
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                probabilities.extend(probs)

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)

    clear_gpu_memory()

    if num_labels == 2:
        return np.array(predictions), np.array(probabilities)
    return np.array(predictions), None


TRAINING mBERT MODELS (Memory-Optimized)


In [27]:
# Train mBERT Binary
pred_mbert_bin, scores_mbert_bin = train_mbert_optimized(
    train_comments, y_train_bin, test_comments, y_test_bin,
    num_labels=2, model_name='mBERT Binary', epochs=2
)

# Check if scores_mbert_bin contains NaNs to prevent plotting errors
if np.isnan(scores_mbert_bin).any():
    print("Warning: mBERT binary classification scores contain NaN values. Skipping PR and ROC curve plotting.")
    effective_scores_mbert_bin = None
else:
    effective_scores_mbert_bin = scores_mbert_bin

results_binary['mBERT'] = evaluate_model(
    y_test_bin, pred_mbert_bin, 'mBERT Binary',
    ['Non-Offensive', 'Offensive'], y_scores=effective_scores_mbert_bin
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training mBERT Binary...
Epoch 1/2, Loss: 0.6180
Epoch 2/2, Loss: 0.4933

mBERT Binary - Binary Classification
Accuracy: 0.8076

Classification Report:
               precision    recall  f1-score   support

Non-Offensive       0.83      0.87      0.85       896
    Offensive       0.77      0.70      0.74       554

     accuracy                           0.81      1450
    macro avg       0.80      0.79      0.79      1450
 weighted avg       0.81      0.81      0.81      1450



In [28]:
# Train mBERT Multiclass
pred_mbert_multi, _ = train_mbert_optimized(
    train_comments, y_train_multi, test_comments, y_test_multi,
    num_labels=4, model_name='mBERT Multiclass', epochs=2
)
evaluate_model(
    y_test_multi, pred_mbert_multi, 'mBERT Multiclass',
    ['NO', 'OO', 'OR', 'OS'], is_binary=False
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training mBERT Multiclass...
Epoch 1/2, Loss: 0.8125
Epoch 2/2, Loss: 0.6415

mBERT Multiclass - Multiclass Classification
Accuracy: 0.7717

Classification Report:
              precision    recall  f1-score   support

          NO       0.79      0.92      0.85       896
          OO       0.72      0.60      0.66       486
          OR       0.00      0.00      0.00        49
          OS       0.00      0.00      0.00        19

    accuracy                           0.77      1450
   macro avg       0.38      0.38      0.38      1450
weighted avg       0.73      0.77      0.75      1450



0.7717241379310344

In [29]:
# Performance Comparison Plots
print("\n" + "="*60)
print("GENERATING COMPARISON PLOTS")
print("="*60)

# Binary Classification Comparison
plt.figure(figsize=(12, 6))
models = list(results_binary.keys())
accuracies = list(results_binary.values())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = plt.bar(models, accuracies, color=colors, alpha=0.8, edgecolor='black')
plt.title('Binary Classification - Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy', fontsize=12)
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('mbert-28dec/comparison/binary_accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

# Per-class F1 Comparison
models_f1 = ['LogisticRegression', 'SVM', 'RandomForest', 'mBERT']
f1_scores = []
for model_name in models_f1:
    if model_name == 'LogisticRegression':
        preds = pred_lr
    elif model_name == 'SVM':
        preds = pred_svm
    elif model_name == 'RandomForest':
        preds = pred_rf
    else:
        preds = pred_mbert_bin

    f1 = f1_score(y_test_bin, preds, average=None)
    f1_scores.append(f1)

x = np.arange(2)
width = 0.2
fig, ax = plt.subplots(figsize=(10, 6))
for i, (model, scores) in enumerate(zip(models_f1, f1_scores)):
    ax.bar(x + i*width, scores, width, label=model, alpha=0.8)

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['Non-Offensive', 'Offensive'])
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('mbert-28dec/comparison/per_class_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.close()


GENERATING COMPARISON PLOTS


In [30]:
# Final Results Table
print("\n" + "="*80)
print("FINAL RESULTS TABLE")
print("="*80)

# Calculate metrics for all binary models
binary_results = {}

models_config = [
    ('Baseline', lr, pred_lr),
    ('LR', lr, pred_lr),
    ('SVM', svm, pred_svm),
    ('RF', rf_bin, pred_rf),
    ('M-BERT', None, pred_mbert_bin)
]

for model_name, model_obj, predictions in models_config:
    # Calculate per-class metrics
    from sklearn.metrics import precision_recall_fscore_support
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test_bin, predictions, average=None, zero_division=0
    )

    binary_results[model_name] = {
        'Non-Offensive': {'Prec': prec[0], 'Rec': rec[0], 'F1': f1[0]},
        'Offensive': {'Prec': prec[1], 'Rec': rec[1], 'F1': f1[1]}
    }

# Create formatted table
print("\n┌─────────────┬────────────────────────────┬────────────────────────────┐")
print("│   Models    │       Non-Offensive        │        Offensive           │")
print("├─────────────┼─────────┬─────────┬────────┼─────────┬─────────┬────────┤")
print("│             │  Prec.  │  Rec.   │   F₁   │  Prec.  │  Rec.   │   F₁   │")
print("├─────────────┼─────────┼─────────┼────────┼─────────┼─────────┼────────┤")

for model_name in ['Baseline', 'LR', 'SVM', 'RF', 'M-BERT']:
    metrics = binary_results[model_name]
    no_prec = metrics['Non-Offensive']['Prec']
    no_rec = metrics['Non-Offensive']['Rec']
    no_f1 = metrics['Non-Offensive']['F1']
    off_prec = metrics['Offensive']['Prec']
    off_rec = metrics['Offensive']['Rec']
    off_f1 = metrics['Offensive']['F1']

    print(f"│ {model_name:11s} │  {no_prec:.2f}   │  {no_rec:.2f}   │  {no_f1:.2f}  │  {off_prec:.2f}   │  {off_rec:.2f}   │  {off_f1:.2f}  │")

print("└─────────────┴─────────┴─────────┴────────┴─────────┴─────────┴────────┘")

# Save results to CSV
results_df = pd.DataFrame([
    {
        'Model': model_name,
        'Non-Offensive_Precision': f"{metrics['Non-Offensive']['Prec']:.2f}",
        'Non-Offensive_Recall': f"{metrics['Non-Offensive']['Rec']:.2f}",
        'Non-Offensive_F1': f"{metrics['Non-Offensive']['F1']:.2f}",
        'Offensive_Precision': f"{metrics['Offensive']['Prec']:.2f}",
        'Offensive_Recall': f"{metrics['Offensive']['Rec']:.2f}",
        'Offensive_F1': f"{metrics['Offensive']['F1']:.2f}"
    }
    for model_name, metrics in binary_results.items()
])

results_df.to_csv('mbert-28dec/final_results.csv', index=False)
print(f"\n✓ Results table saved to: mbert-28dec/final_results.csv")


FINAL RESULTS TABLE

┌─────────────┬────────────────────────────┬────────────────────────────┐
│   Models    │       Non-Offensive        │        Offensive           │
├─────────────┼─────────┬─────────┬────────┼─────────┬─────────┬────────┤
│             │  Prec.  │  Rec.   │   F₁   │  Prec.  │  Rec.   │   F₁   │
├─────────────┼─────────┼─────────┼────────┼─────────┼─────────┼────────┤
│ Baseline    │  0.78   │  0.91   │  0.84  │  0.80   │  0.58   │  0.67  │
│ LR          │  0.78   │  0.91   │  0.84  │  0.80   │  0.58   │  0.67  │
│ SVM         │  0.79   │  0.89   │  0.83  │  0.77   │  0.61   │  0.68  │
│ RF          │  0.79   │  0.93   │  0.85  │  0.83   │  0.61   │  0.70  │
│ M-BERT      │  0.83   │  0.87   │  0.85  │  0.77   │  0.70   │  0.74  │
└─────────────┴─────────┴─────────┴────────┴─────────┴─────────┴────────┘

✓ Results table saved to: mbert-28dec/final_results.csv


In [31]:
# Create visual table as image
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')

table_data = [['Models', 'Non-Offensive', '', '', 'Offensive', '', ''],
              ['', 'Prec.', 'Rec.', 'F₁', 'Prec.', 'Rec.', 'F₁']]

for model_name in ['Baseline', 'LR', 'SVM', 'RF', 'M-BERT']:
    metrics = binary_results[model_name]
    row = [
        model_name,
        f"{metrics['Non-Offensive']['Prec']:.2f}",
        f"{metrics['Non-Offensive']['Rec']:.2f}",
        f"{metrics['Non-Offensive']['F1']:.2f}",
        f"{metrics['Offensive']['Prec']:.2f}",
        f"{metrics['Offensive']['Rec']:.2f}",
        f"{metrics['Offensive']['F1']:.2f}"
    ]
    table_data.append(row)

table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                colWidths=[0.15, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header rows
for i in range(7):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(weight='bold', color='white')
    table[(1, i)].set_facecolor('#8EAADB')
    table[(1, i)].set_text_props(weight='bold')

# Merge cells for headers
table[(0, 1)].set_text_props(text='Non-Offensive')
table[(0, 1)].visible_edges = 'LRTB'
table[(0, 2)].visible_edges = ''
table[(0, 3)].visible_edges = ''
table[(0, 4)].set_text_props(text='Offensive')
table[(0, 4)].visible_edges = 'LRTB'
table[(0, 5)].visible_edges = ''
table[(0, 6)].visible_edges = ''

plt.title('Binary Classification Results - Per-Class Metrics',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('mbert-28dec/final_results_table.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Results table image saved to: mbert-28dec/final_results_table.png")

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"\nAll results saved in 'mbert-28dec/' directory:")
print(f"  ✓ Confusion matrices: mbert-28dec/confusion_matrices/")
print(f"  ✓ PR curves: mbert-28dec/pr_curves/")
print(f"  ✓ ROC curves: mbert-28dec/roc_curves/")
print(f"  ✓ Comparison plots: mbert-28dec/comparison/")
print(f"  ✓ Final results: mbert-28dec/final_results.csv")
print(f"  ✓ Final results table: mbert-28dec/final_results_table.png")
print("="*80)


✓ Results table image saved to: mbert-28dec/final_results_table.png

TRAINING COMPLETE!

All results saved in 'mbert-28dec/' directory:
  ✓ Confusion matrices: mbert-28dec/confusion_matrices/
  ✓ PR curves: mbert-28dec/pr_curves/
  ✓ ROC curves: mbert-28dec/roc_curves/
  ✓ Comparison plots: mbert-28dec/comparison/
  ✓ Final results: mbert-28dec/final_results.csv
  ✓ Final results table: mbert-28dec/final_results_table.png
